# LLM-AGR Reproduction — Amazon-book & Yelp

**Run all cells top-to-bottom at the start of every session.**  
The grid cell skips already-completed runs, so sessions are safe to resume.

Before running:
1. Set `GITHUB_REPO` in Cell 1 to your repo URL.
2. Upload both dataset folders to `MyDrive/LLM-AGR-data/` (amazon/ and yelp/ pkl files, ~550 MB total).
3. Optionally upload existing smoke-test logs to `MyDrive/LLM-AGR-results/logs/` to skip those 2 runs.
4. Make sure the runtime is set to **GPU** (Runtime → Change runtime type → T4 GPU).

In [ ]:
# ── Cell 1: Mount Drive & clone code ──────────────────────────────────────────
GITHUB_REPO = "https://github.com/mertrodop/CS_555_Project.git"

from google.colab import drive
drive.mount('/content/drive')

import os
if not os.path.exists('/content/LLM-AGR'):
    os.system(f'git clone {GITHUB_REPO} /content/LLM-AGR')
else:
    print('Repo already cloned — skipping clone')

# Verify GPU is available
import torch
assert torch.cuda.is_available(), "No GPU found — check Runtime > Change runtime type"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"PyTorch: {torch.__version__}  CUDA: {torch.version.cuda}")

In [ ]:
# ── Cell 2: Copy data from Drive to local /content (fast SSD) ─────────────────
import shutil, os

DATA_DRIVE = '/content/drive/MyDrive/LLM-AGR-data'
DATA_LOCAL = '/content/LLM-AGR/data'
os.makedirs(DATA_LOCAL, exist_ok=True)

for ds in ['amazon', 'yelp']:
    src = f'{DATA_DRIVE}/{ds}'
    dst = f'{DATA_LOCAL}/{ds}'
    assert os.path.exists(src), f"Missing {src} — upload data to Drive first"
    if not os.path.exists(dst):
        shutil.copytree(src, dst)
        print(f'Copied {ds} data to {dst}')
    else:
        print(f'{ds} data already present — skipping copy')

In [ ]:
# ── Cell 3: Install dependencies ──────────────────────────────────────────────
import os, torch

# Standard packages (pyyaml, scipy, tqdm, pandas)
os.system('pip install -r /content/LLM-AGR/requirements.txt -q')

# PyG packages need version-specific wheel URL — auto-detect from Colab's runtime
torch_ver = torch.__version__.split('+')[0]
cuda_ver  = torch.version.cuda.replace('.', '')
pyg_url   = f'https://data.pyg.org/whl/torch-{torch_ver}+cu{cuda_ver}.html'
print(f'Installing torch_sparse + torch_scatter from:\n  {pyg_url}')
os.system(f'pip install torch_sparse torch_scatter -f {pyg_url} -q')

# Verify
import torch_sparse, torch_scatter
print('All dependencies installed successfully')

In [ ]:
# ── Cell 4: Restore completed logs from Drive (makes grid resumable) ───────────
import shutil, os

DRIVE_LOGS = '/content/drive/MyDrive/LLM-AGR-results/logs'
LOCAL_LOGS = '/content/LLM-AGR/logs'
os.makedirs(LOCAL_LOGS, exist_ok=True)

if os.path.exists(DRIVE_LOGS):
    shutil.copytree(DRIVE_LOGS, LOCAL_LOGS, dirs_exist_ok=True)
    n = sum(1 for _, _, fs in os.walk(LOCAL_LOGS) for f in fs if f.endswith('.log'))
    print(f'Restored {n} log(s) from Drive — those runs will be skipped')
else:
    print('No existing logs on Drive — starting fresh')

In [ ]:
%%writefile /content/LLM-AGR/run_grid.sh
#!/bin/bash
set -e
cd /content/LLM-AGR
DRIVE_LOGS="/content/drive/MyDrive/LLM-AGR-results/logs"

for model in lightgcn lightgcn_agr sgl sgl_agr simgcl simgcl_agr bigcf bigcf_agr; do
  for dataset in amazon yelp; do
    for seed in 0 1 2 3 4; do
      LOG="logs/${dataset}/${model}_seed${seed}.log"
      mkdir -p "logs/${dataset}"
      if [ -f "$LOG" ]; then
        echo "[SKIP] $LOG already exists"
        continue
      fi
      echo "====== ${model} | ${dataset} | seed=${seed} ======"
      python main.py --model $model --dataset $dataset --seed $seed \
        2>&1 | tee "$LOG"
      # Immediately sync this log to Drive so a session reset doesn't lose it
      mkdir -p "${DRIVE_LOGS}/${dataset}"
      cp "$LOG" "${DRIVE_LOGS}/${dataset}/"
    done
  done
done
echo "Grid complete."

In [ ]:
# ── Cell 5b: Run the grid (this cell will run for several hours) ───────────────
# Each completed log is synced to Drive immediately, so you can interrupt
# and resume in a new session without losing progress.
os.makedirs('/content/drive/MyDrive/LLM-AGR-results/logs', exist_ok=True)
os.system('bash /content/LLM-AGR/run_grid.sh')

In [ ]:
# ── Cell 6: Final sync — run any time to push all logs & results to Drive ──────
import shutil, os

DRIVE_OUT = '/content/drive/MyDrive/LLM-AGR-results'
os.makedirs(DRIVE_OUT, exist_ok=True)

for folder in ['logs', 'results', 'checkpoint']:
    src = f'/content/LLM-AGR/{folder}'
    dst = f'{DRIVE_OUT}/{folder}'
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f'Synced {folder}/ → Drive')

# Count completed runs
logs_dir = f'{DRIVE_OUT}/logs'
n = sum(1 for _, _, fs in os.walk(logs_dir) for f in fs if f.endswith('.log'))
print(f'\n{n}/80 runs completed so far')

## After All 80 Runs — Run Aggregation

Once all logs are collected, run the `aggregate.py` script (committed to your repo) to produce:
- `results/amazon_book.csv`, `results/yelp.csv`
- `results/table3_repro.md` (mean ± std, significance markers)
- `results/paper_vs_ours.md` (gap to paper numbers)
- `results/SUMMARY.md`

In [ ]:
# ── Cell 7 (run after all 80 logs exist): Aggregate results ───────────────────
os.chdir('/content/LLM-AGR')
os.makedirs('results', exist_ok=True)
os.system('python aggregate.py')
# Sync final results to Drive
shutil.copytree('results', f'{DRIVE_OUT}/results', dirs_exist_ok=True)
print('Aggregation complete — results synced to Drive')